# Test all V3 checkpoints: Table 2, Table 3, and Table 4

The notebook scans every EMA/Raw checkpoint in the selected Kaggle Dataset folder. Each checkpoint is evaluated independently with threshold `0.5`. GPT-3.5 Mixed 0% is excluded because it is already evaluated after training.


In [ ]:
from pathlib import Path
import re, shutil, subprocess

# CONFIG: set CHECKPOINT_FOLDER to your Kaggle Dataset directory.
BRANCH='version3'
REPO_URL='https://github.com/tphan1420/PRDetect-Research.git'
WORK=Path('/kaggle/working/v3_all_checkpoints_table2_table3_table4')
REPO=WORK/'PRDetect-Research'
CHECKPOINT_FOLDER=Path('/kaggle/input/datasets/your-account/your-v3-dual-checkpoints')
STRUCT_MODE='low'
BATCH_SIZE=4

WORK.mkdir(parents=True,exist_ok=True)
assert CHECKPOINT_FOLDER.is_dir(), f'Checkpoint folder does not exist: {CHECKPOINT_FOLDER}'

# Scan checkpoints recursively. A seed can appear anywhere in the filename,
# for example: v3_seed2026_b64_raw_for_eval.pt.
paths=sorted(p for p in CHECKPOINT_FOLDER.rglob('*')
             if p.is_file() and p.suffix.lower() in {'.pt','.pth'})
assert paths, f'No .pt/.pth files in: {CHECKPOINT_FOLDER}'
CHECKPOINTS=[]
used_case_ids=set()
for checkpoint in paths:
    match=re.search(r'(?<!\d)(20\d{2})(?!\d)', checkpoint.stem)
    assert match, f'No 20xx seed in checkpoint filename: {checkpoint.name}'
    seed=int(match.group(1))
    # case_id is derived from the checkpoint name; add a suffix only for duplicate stems.
    base=re.sub(r'[^A-Za-z0-9._-]+','_',checkpoint.stem).strip('._') or 'checkpoint'
    case_id=base
    suffix=2
    while case_id in used_case_ids:
        case_id=f'{base}__{suffix}'; suffix+=1
    used_case_ids.add(case_id)
    CHECKPOINTS.append(dict(path=checkpoint,seed=seed,case_id=case_id))

if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO)],check=True)
assert (REPO/'evaluate_v3_quang.py').is_file()
COMMIT=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
print({'checkpoint_folder':str(CHECKPOINT_FOLDER),'count':len(CHECKPOINTS),
       'branch':BRANCH,'commit':COMMIT})
for item in CHECKPOINTS:
    print(f"- {item['case_id']} | seed={item['seed']} | {item['path']}")


In [ ]:
%cd {REPO}
!pip install -q -r requirements-v3-eval.txt
!python -m spacy download en_core_web_sm
assert _exit_code == 0, 'Cài đặt lỗi; hãy bật Internet và GPU.'
!python -c "import torch; assert torch.cuda.is_available(), 'Hãy bật GPU'; print(torch.__version__, torch.cuda.get_device_name(0))"
assert _exit_code == 0


In [ ]:
# (suite, condition, source JSON relative to REPO)
# Comment out a row to skip that test file.
CASES = [
    # Table 2: train HC3 -> test HC3 (in-domain lexical noise).
    ('table2', 'Original',     0, 'data/original_text/benchmarks/hc3/hc3_test.json'),
    ('table2', 'Synonym 5%',   5, 'data/perturbed_text/hc3/lexical_synonym/hc3_test_adj_5_synonym_replace.json'),
    ('table2', 'Synonym 10%', 10, 'data/perturbed_text/hc3/lexical_synonym/hc3_test_adj_10_synonym_replace.json'),
    ('table2', 'Synonym 20%', 20, 'data/perturbed_text/hc3/lexical_synonym/hc3_test_adj_20_synonym_replace.json'),
    ('table2', 'Synonym 30%', 30, 'data/perturbed_text/hc3/lexical_synonym/hc3_test_adj_30_synonym_replace.json'),

    # Table 3: train HC3 -> GPT-3.5 Mixed (cross-dataset lexical noise).
    ('table3', 'Synonym 5%',  5, 'data/perturbed_text/gpt35_mixed/lexical_synonym/gpt3.5_mixed_test_adj_5_synonym_replace.json'),
    ('table3', 'Synonym 10%', 10, 'data/perturbed_text/gpt35_mixed/lexical_synonym/gpt3.5_mixed_test_adj_10_synonym_replace.json'),
    ('table3', 'Synonym 20%', 20, 'data/perturbed_text/gpt35_mixed/lexical_synonym/gpt3.5_mixed_test_adj_20_synonym_replace.json'),
    ('table3', 'Synonym 30%', 30, 'data/perturbed_text/gpt35_mixed/lexical_synonym/gpt3.5_mixed_test_adj_30_synonym_replace.json'),

    # Table 4: HC3 structural noise.
    ('table4', 'Original',     0, 'data/original_text/benchmarks/hc3/hc3_test.json'),
    ('table4', 'Insert 25%',  25, 'data/perturbed_text/hc3/structural_noise/hc3_test_insert_25.json'),
    ('table4', 'Repeat 25%',  25, 'data/perturbed_text/hc3/structural_noise/hc3_test_repeat_25.json'),
    ('table4', 'Replace 25%', 25, 'data/perturbed_text/hc3/structural_noise/hc3_test_replace_25.json'),
    ('table4', 'Delete 25%',  25, 'data/perturbed_text/hc3/structural_noise/hc3_test_delete_25.json'),
]
assert CASES, 'Ph?i gi? ?t nh?t m?t file test.'
for suite, condition, noise, source in CASES:
    assert (REPO/source).is_file(), source
    print(f'{suite}: {condition} ? {source}')


In [ ]:
%%writefile /kaggle/working/v3_all_checkpoints_table2_table3_table4/evaluate_all_selected_v3.py
import argparse, json, shutil, subprocess, sys
from dataclasses import asdict
from pathlib import Path

# The notebook launches this script with cwd=REPO, so imports and graph building
# use exactly the freshly cloned version3 source tree.
sys.path.insert(0, str(Path.cwd()))

parser=argparse.ArgumentParser()
parser.add_argument('--checkpoints-json',type=Path,required=True)
parser.add_argument('--cases-json',type=Path,required=True)
parser.add_argument('--struct-mode',choices=('low','high'),default='low')
parser.add_argument('--batch-size',type=int,default=4)
parser.add_argument('--work-dir',type=Path,required=True)
args=parser.parse_args()

import evaluate_v3_quang as ev
work=args.work_dir.resolve(); work.mkdir(parents=True,exist_ok=True)
raw_checkpoints=json.loads(args.checkpoints_json.read_text(encoding='utf-8'))
raw_cases=json.loads(args.cases_json.read_text(encoding='utf-8'))
assert raw_checkpoints, 'No checkpoints selected.'
cases=[ev.Case(*item) for item in raw_cases]

# One graph per unique source. Sources with identical split names must also be
# identical files, otherwise building_graph.py would overwrite a different graph.
graphs={}
for case in cases:
    source=(ev.REPO/case.source).resolve()
    if not source.is_file():
        raise FileNotFoundError(source)
    previous=graphs.setdefault(case.split,source)
    if previous != source:
        raise ValueError(f'Graph-name collision for {case.split}: {previous} vs {source}')

def expected_samples(source):
    with source.open(encoding='utf-8') as stream:
        return sum(bool(line.strip()) for line in stream)

# Build all test graphs exactly once in the cloned repository. The cache is
# shared by every checkpoint and removed only after the full evaluation loop.
graph_paths={}
for split,source in graphs.items():
    graph=ev.REPO/'graph_data'/f'{split}.pkl'
    graph.unlink(missing_ok=True)
    print(f'Build once: {split} ({expected_samples(source)} samples)',flush=True)
    subprocess.run([sys.executable,str(ev.REPO/'building_graph.py'),'--file',str(source)],
                   cwd=ev.REPO,check=True)
    if not graph.is_file():
        raise FileNotFoundError(graph)
    graph_paths[split]=graph

all_runs=[]
try:
    for item in raw_checkpoints:
        checkpoint=Path(item['path']).resolve()
        seed=int(item['seed'])
        case_id=str(item['case_id'])
        if not checkpoint.is_file():
            raise FileNotFoundError(checkpoint)
        run_dir=work/'runs'/case_id
        run_dir.mkdir(parents=True,exist_ok=True)
        staged=run_dir/'staged_checkpoint'; staged.mkdir(exist_ok=True)
        alias=staged/f'v3_baseline_seed{seed}.pt'
        shutil.copy2(checkpoint,alias)

        config_args=argparse.Namespace(
            variant='baseline',train_domain='hc3',checkpoint_dir=str(staged),seeds=[seed],
            batch_size=args.batch_size,struct_mode=args.struct_mode,device='cuda',
            graph_budget_root=None,
        )
        config=ev.make_config(config_args)
        actual=Path(config['checkpoint_paths'][str(seed)]).resolve()
        if actual != alias.resolve():
            raise ValueError(f'Checkpoint resolution mismatch: requested={alias}, found={actual}')
        config_path=run_dir/'config.json'
        ev.write_json(config_path,config)
        results=run_dir/'result'; results.mkdir(exist_ok=True)
        ev.run_child('inspect','--config',config_path,'--output',results/'checkpoint_metadata.json')

        print(f'\\n=== {case_id} | seed={seed} ===',flush=True)
        for case in cases:
            source=graphs[case.split]
            expected=expected_samples(source)
            signature=ev.case_signature(config,case,expected)
            report=ev.report_path(results,case,seed)
            ev.run_child('_test','--config',config_path,'--seed',seed,
                         '--graph',graph_paths[case.split],'--report',report,
                         '--expected-samples',expected,'--signature',signature)

        ev.summarize(cases,results,[seed])
        manifest={**config,'suite':'selected_table2_table3_table4',
                  'cases':[asdict(case) for case in cases],
                  'checkpoint_requested':str(checkpoint),'fixed_threshold':0.5,
                  'graphs_built_once':True,
                  'table4_average_included':len([c for c in cases if c.suite=='table4' and c.noise_percent==25])==4}
        ev.write_json(results/'manifest.json',manifest)
        archive=run_dir/f'v3_baseline_selected_table2_table3_table4_seed{seed}.zip'
        ev.export_archive(results,archive,cases,[seed])
        all_runs.append(dict(case_id=case_id,seed=seed,checkpoint=str(checkpoint),
                             run_dir=str(run_dir),archive=str(archive)))
finally:
    for graph in graph_paths.values():
        graph.unlink(missing_ok=True)

manifest_path=work/'all_checkpoints_manifest.json'
ev.write_json(manifest_path,all_runs)
print('All checkpoints complete:',len(all_runs),flush=True)


In [ ]:
import json, subprocess, sys
CASES_JSON=WORK/'selected_cases.json'
CASES_JSON.write_text(json.dumps(CASES,ensure_ascii=False,indent=2),encoding='utf-8')
CHECKPOINTS_JSON=WORK/'selected_checkpoints.json'
CHECKPOINTS_JSON.write_text(json.dumps([
    dict(path=str(item['path']),seed=item['seed'],case_id=item['case_id'])
    for item in CHECKPOINTS
],ensure_ascii=False,indent=2),encoding='utf-8')

RUNNER=WORK/'evaluate_all_selected_v3.py'
assert RUNNER.is_file(), f'Run the runner cell first: {RUNNER}'
subprocess.run([
    sys.executable,str(RUNNER),'--checkpoints-json',str(CHECKPOINTS_JSON),
    '--cases-json',str(CASES_JSON),'--struct-mode',STRUCT_MODE,
    '--batch-size',str(BATCH_SIZE),'--work-dir',str(WORK),
],cwd=REPO,check=True)

RUN_MANIFEST=WORK/'all_checkpoints_manifest.json'
assert RUN_MANIFEST.is_file(), RUN_MANIFEST
ALL_RUNS=json.loads(RUN_MANIFEST.read_text(encoding='utf-8'))
assert len(ALL_RUNS)==len(CHECKPOINTS)


In [ ]:
from IPython.display import FileLink, display
import zipfile

assert ALL_RUNS, 'Ch?y cell ??nh gi? checkpoint tr??c.'
archive=WORK/'v3_baseline_all_checkpoints_table2_table3_table4.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as output:
    output.write(RUN_MANIFEST,RUN_MANIFEST.name)
    for item in ALL_RUNS:
        path=Path(item['archive'])
        output.write(path,f"{item['case_id']}/{path.name}")
print('ZIP:',archive)
display(FileLink(str(archive)))
